# 7 · Word-level recognition (IPA & orthographic)  (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/07_word_level/07_word_level_recognition.ipynb)

An **alternative recognition target** to the phoneme model (Notebook 5). Trains
a **character-level** `Wav2Vec2CTCTokenizer` model (space → `|` word delimiter)
on whole words; reported with **WER / CER**. Reuses the same backbone and the
same manifest — only the label and vocabulary change.

Pick the target with **`TARGET`**:

| `TARGET` | label column | example output |
|---|---|---|
| `"ipa"` | word-form IPA (`ipa_wordform`) | `dat hœʁt` |
| `"orthography"` | Kölsch spelling (`text`) | `dat hührt` |

> Run on a CUDA GPU (Colab → Runtime → GPU). A single wrong character fails the
> whole word, so word-level WER sits above the phoneme error rate — expected.

## Setup

In [ ]:
!pip -q install "transformers>=4.40" datasets evaluate jiwer torchaudio accelerate librosa soundfile
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # harmless on GPU
import torch, json, re, numpy as np
from dataclasses import dataclass
from typing import Union
USE_CUDA = torch.cuda.is_available()
device = "cuda" if USE_CUDA else "cpu"
print("torch", torch.__version__, "·", device)
if not USE_CUDA:
    print("NOTE: no CUDA GPU — on Colab set Runtime -> GPU. CPU/MPS is very slow here.")

In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import sys
from pathlib import Path
try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/kolsch-tandem.git /content/kolsch-tandem")
except Exception:
    pass
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, PAGES, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)
print("repo root:", ROOT)

## 1 · Choose target, load manifest, split

In [ ]:
import pandas as pd, hashlib

TARGET = "ipa"            # "ipa"  or  "orthography"
LABEL_COL = {"ipa": "ipa_wordform", "orthography": "text"}[TARGET]

MANIFEST = os.path.join(SEG, "manifest.csv")
assert os.path.exists(MANIFEST), f"{MANIFEST} not found — run Notebooks 3 & 4 first."
man = pd.read_csv(MANIFEST)
assert LABEL_COL in man.columns, f"manifest has no '{LABEL_COL}' column — run Notebook 4."
man = man.merge(pd.read_csv(INDEX)[["id","speaker"]], on="id", how="left")

def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"\d+", "", s)
    s = re.sub(r"[^a-zäöüßçæœø' \t]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

man["label"] = man[LABEL_COL].map(normalize_text) if TARGET == "orthography" else man[LABEL_COL]

def bucket(sp):
    h = int(hashlib.md5(str(sp).encode()).hexdigest(), 16) % 10
    return "test" if h < 1 else "valid" if h < 2 else "train"
speakers = man["speaker"].fillna("unknown").unique()
if len(speakers) >= 3:
    man["split"] = man["speaker"].map(bucket)
else:
    rng = np.random.default_rng(0); r = rng.random(len(man))
    man["split"] = np.where(r < 0.8, "train", np.where(r < 0.9, "valid", "test"))
if len(man) > 1 and (man["split"] == "valid").sum() == 0:
    man.loc[man.index[-1], "split"] = "valid"
print("TARGET =", TARGET, "| rows:", len(man),
      "| split:", {s:int((man["split"]==s).sum()) for s in ["train","valid","test"]})

## 2 · Character vocabulary + processor (space → `|`)

In [ ]:
from datasets import Dataset
from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor)

def extract_chars(series):
    chars = set()
    for s in series.astype(str): chars.update(s.replace(" ", ""))
    return chars

VOCAB_PATH = os.path.join(MODELS, f"vocab_wordlevel_{TARGET}.json")
def build_char_vocab(train_labels, path=VOCAB_PATH):
    vocab = {c: i for i, c in enumerate(sorted(extract_chars(train_labels)))}
    vocab["|"] = len(vocab); vocab["[UNK]"] = len(vocab); vocab["[PAD]"] = len(vocab)
    json.dump(vocab, open(path, "w"), ensure_ascii=False)
    return vocab

vocab = build_char_vocab(man[man["split"] == "train"]["label"])
tokenizer = Wav2Vec2CTCTokenizer(VOCAB_PATH, unk_token="[UNK]", pad_token="[PAD]",
                                 word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000,
                padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
print(f"{TARGET} char vocab: {len(vocab)} tokens ->", VOCAB_PATH)

## 3 · Datasets + prepare (load audio manually — no Arrow Audio cast)

In [ ]:
import soundfile as sf
try:
    import librosa
except Exception:
    librosa = None

def to_ds(split):
    sub = man[man["split"] == split][["audio_path", "label"]]
    return Dataset.from_pandas(sub, preserve_index=False)

def _load_16k(path):
    wav, sr = sf.read(path)
    if getattr(wav, "ndim", 1) > 1: wav = wav.mean(axis=1)
    wav = np.asarray(wav, dtype=np.float32)
    if sr != 16000:
        if librosa is None: raise RuntimeError("pip install librosa to resample")
        wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    return wav

def prepare(batch):
    wav = _load_16k(batch["audio_path"])
    batch["input_values"] = processor(wav, sampling_rate=16000).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor(text=batch["label"]).input_ids     # space -> |
    return batch

train_ds = to_ds("train").map(prepare, remove_columns=["audio_path","label"])
valid_ds = to_ds("valid").map(prepare, remove_columns=["audio_path","label"])
print("prepared:", len(train_ds), "train /", len(valid_ds), "valid")

## 4 · Collator + metrics

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids":   f["labels"]}        for f in features]
        batch = self.processor.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch
data_collator = DataCollatorCTCWithPadding(processor=processor)

import evaluate
wer_metric, cer_metric = evaluate.load("wer"), evaluate.load("cer")
def compute_metrics(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    ids = np.argmax(logits, axis=-1)
    labels = pred.label_ids; labels[labels == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(ids)                        # "dat hœʁt"
    label_str = processor.batch_decode(labels, group_tokens=False)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str),
            "cer": cer_metric.compute(predictions=pred_str, references=label_str)}

## 5 · Model + GPU training config

In [ ]:
from transformers import Wav2Vec2ForCTC, TrainingArguments, Trainer
import inspect

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    ctc_loss_reduction="mean", ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id, vocab_size=len(processor.tokenizer))
model.freeze_feature_encoder()
model.gradient_checkpointing_enable()

# desired args, filtered to what THIS transformers version supports
_want = dict(
    output_dir=os.path.join(MODELS, f"kolsch_wordlevel_{TARGET}"),
    group_by_length=True,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    per_device_eval_batch_size=8, num_train_epochs=150,
    fp16=USE_CUDA, dataloader_pin_memory=USE_CUDA,
    learning_rate=3e-5, lr_scheduler_type="cosine", warmup_steps=500, weight_decay=0.05,
    eval_strategy="steps", save_strategy="steps", eval_steps=1000, save_steps=1000,
    logging_steps=100, load_best_model_at_end=True,
    metric_for_best_model="wer", greater_is_better=False, save_total_limit=2, report_to="none")
_sig = set(inspect.signature(TrainingArguments.__init__).parameters)
if "eval_strategy" not in _sig and "evaluation_strategy" in _sig:
    _want["evaluation_strategy"] = _want.pop("eval_strategy")
_dropped = [k for k in _want if k not in _sig]
args = TrainingArguments(**{k: v for k, v in _want.items() if k in _sig})
if _dropped: print("note: transformers version ignores:", _dropped)
print("model + args ready")

## 6 · Train + save

In [ ]:
trainer_kwargs = dict(model=model, args=args, data_collator=data_collator,
                      train_dataset=train_ds, eval_dataset=valid_ds,
                      compute_metrics=compute_metrics)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = processor.feature_extractor
else:
    trainer_kwargs["tokenizer"] = processor.feature_extractor
trainer = Trainer(**trainer_kwargs)

trainer.train()

out = os.path.join(MODELS, f"kolsch_wordlevel_{TARGET}")
trainer.save_model(out); processor.save_pretrained(out)
print("saved ->", out)

## What to expect

Run twice (`TARGET="ipa"`, then `"orthography"`) to get both models, and compare
against the phoneme recogniser (Notebook 5) on the same test split. Word-level
WER is higher than the phoneme error rate because one wrong character fails the
whole word — arithmetic, not a regression.